# 工具高级特性

本节介绍 `return_direct`、`InjectedToolCallId`、`ToolException` 和错误处理等高级特性。

In [1]:
from dotenv import load_dotenv
load_dotenv()

print("环境变量已加载")

环境变量已加载


## return_direct——直接返回最终结果

默认工具结果会返回给模型再加工。`return_direct=True` 时，工具结果直接作为最终输出，跳过模型二次加工。

In [2]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

# 普通工具：结果返回给模型再做总结
@tool
def search_normal(keyword: str) -> str:
    """搜索菜鸟教程 RUNOOB 的课程（普通模式）"""
    return f"搜索结果：Python3 基础教程、Python 数据分析、Python 爬虫入门"

# return_direct 工具：结果直接作为最终输出
@tool(return_direct=True)
def search_direct(keyword: str) -> str:
    """搜索菜鸟教程 RUNOOB 的课程（直接返回模式）。
    当用户只需要搜索结果，不需要额外分析时使用此工具。
    """
    return f"搜索结果：Python3 基础教程、Python 数据分析、Python 爬虫入门"

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent_normal = create_agent(model=model, tools=[search_normal], system_prompt="你是菜鸟教程的学习顾问。")
agent_direct = create_agent(model=model, tools=[search_direct], system_prompt="你是菜鸟教程的学习顾问。")

result = agent_normal.invoke({"messages": [HumanMessage(content="搜索 Python 课程")]})
print("普通模式:", result["messages"][-1].content[:100])

result = agent_direct.invoke({"messages": [HumanMessage(content="搜索 Python 课程")]})
print("直接返回:", result["messages"][-1].content[:100])

普通模式: 🎯 为您找到了以下 **Python 相关课程**，来看看吧！

---

### 📘 1. Python3 基础教程
- **适合人群**：零基础入门、想系统学习 Python 的初学者
- **内
直接返回: 搜索结果：Python3 基础教程、Python 数据分析、Python 爬虫入门


| 模式 | 行为 | 适用场景 |
| --- | --- | --- |
| return_direct=False（默认） | 模型收到结果 → 继续思考 → 生成回复 | 需要分析/总结/决策 |
| return_direct=True | 工具结果直接作为最终输出 | 查询类、已格式化好的结果 |

## InjectedToolCallId——获取工具调用 ID

在工具函数中注入当前的 `tool_call_id`，用于日志记录或追溯。

In [7]:
from typing import Annotated
from langchain.tools import tool, InjectedToolCallId

@tool
def log_user_action(
    action: str,
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> str:
    """记录用户操作到日志系统。

    Args:
        action: 用户操作描述
        tool_call_id: 系统自动注入的工具调用 ID
    """
    return f"操作已记录 (调用ID: {tool_call_id}): {action}"

# InjectedToolCallId 需要完整的 ToolCall 格式（含 tool_call_id）
result = log_user_action.invoke({
    "args": {"action": "用户查询了 Python 课程"},
    "name": "log_user_action",
    "type": "tool_call",
    "id": "call_001",
})
print(result)


content='操作已记录 (调用ID: call_001): 用户查询了 Python 课程' name='log_user_action' tool_call_id='call_001'


> 带 `InjectedToolArg` 标记的参数由框架自动注入，Agent 不会看到这些参数。

## ToolException——工具异常处理

抛出 `ToolException` 让 Agent 知道工具执行出错。

In [8]:
from langchain.tools import tool, ToolException

@tool
def get_user_info(user_id: int) -> str:
    """根据用户 ID 查询用户信息。

    Args:
        user_id: 用户 ID，必须是正整数
    """
    if user_id <= 0:
        raise ToolException(f"用户 ID 必须为正整数，收到了: {user_id}")

    users = {
        1: "张三（VIP 会员，注册于 2024-01-15）",
        2: "李四（普通用户，注册于 2024-03-20）",
    }
    if user_id not in users:
        raise ToolException(f"未找到 ID 为 {user_id} 的用户")
    return users[user_id]

print(get_user_info.invoke({"user_id": 1}))
try:
    get_user_info.invoke({"user_id": -1})
except ToolException as e:
    print(f"异常: {e}")

try:
    get_user_info.invoke({"user_id": 999})
except ToolException as e:
    print(f"异常: {e}")

张三（VIP 会员，注册于 2024-01-15）
异常: 用户 ID 必须为正整数，收到了: -1
异常: 未找到 ID 为 999 的用户


## handle_tool_errors——自动处理工具错误

配置错误信息如何反馈给模型。

In [10]:
from langchain.tools import tool, ToolException
from langchain.chat_models import init_chat_model

@tool
def get_weather(city: str) -> str:
    """查询指定城市的天气。

    Args:
        city: 城市名称，中文全称
    """
    weather_data = {
        "杭州": "晴，25°C", "北京": "多云，18°C", "上海": "小雨，22°C"
    }
    if city not in weather_data:
        raise ToolException(f"未收录 '{city}'。可用：{', '.join(weather_data.keys())}")
    return f"{city}天气：{weather_data[city]}"

# handle_tool_errors=True 时，错误信息返回给模型
# handle_tool_errors 是 ToolNode 的配置参数，不能直接对工具使用
# ToolException 在 Agent 的 ToolNode 中会自动被捕获
try:
    get_weather.invoke({"city": "北境"})
except ToolException as e:
    print(f"ToolException 被捕获: {e}")
print(f"错误反馈: {result}")

ToolException 被捕获: 未收录 '北境'。可用：杭州, 北京, 上海
错误反馈: content='操作已记录 (调用ID: call_001): 用户查询了 Python 课程' name='log_user_action' tool_call_id='call_001'


| handle_tool_errors | 行为 |
| --- | --- |
| True | 捕获异常，错误信息返回给模型 |
| False | 异常直接上抛，Agent 中断 |
| str | 用自定义字符串替换错误信息 |
| tuple[type] | 只处理指定类型异常 |

## 完整示例——带错误处理的 Agent

In [1]:
from langchain.tools import tool, ToolException
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage


@tool
def book_course(user_name: str, course_name: str) -> str:
    """为用户预订菜鸟教程 RUNOOB 的课程。

    Args:
        user_name: 用户姓名
        course_name: 课程名称
    """
    # 校验用户是否存在
    valid_users = {"张三", "李四", "王五"}
    if user_name not in valid_users:
        raise ToolException(
            f"用户 '{user_name}' 不存在。"
            f"有效用户：{', '.join(sorted(valid_users))}"
        )

    # 校验课程是否存在
    valid_courses = {"Python3 基础教程", "HTML 基础教程", "Java 面向对象"}
    if course_name not in valid_courses:
        raise ToolException(
            f"课程 '{course_name}' 不存在。"
            f"有效课程：{', '.join(sorted(valid_courses))}"
        )

    return f"已为 {user_name} 成功预订《{course_name}》"


model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(
    model=model,
    tools=[book_course],
    system_prompt="你是菜鸟教程 RUNOOB 的课程顾问。",
)

# 正常调用
result = agent.invoke({
    "messages": [HumanMessage(content="帮张三预订 Python3 基础教程")]
})
print(f"成功: {result['messages'][-1].content[:100]}")

# 错误调用：用户不存在
result = agent.invoke({
    "messages": [HumanMessage(content="帮赵六预订 Python3 基础教程")]
})
print(f"\n错误-用户不存在:")
for msg in result["messages"]:
    if msg.type == "tool":
        print(f"  [{msg.type}] {msg.content[:80]}")
    elif msg.type == "ai" and msg.content:
        print(f"  [{msg.type}] {msg.content[:100]}")

成功: 已成功为 **张三** 预订了 **《Python3 基础教程》**！🎉

如需了解更多课程信息或继续预订其他课程，欢迎随时联系我！


ToolException: 用户 '赵六' 不存在。有效用户：张三, 李四, 王五

**术语：**

- **return_direct**：工具结果直接作为最终输出，跳过模型加工
- **InjectedToolCallId**：自动注入的工具调用 ID
- **ToolException**：工具级异常，被 Agent 框架捕获
- **handle_tool_errors**：配置工具错误处理策略